# <p align = "center"> S3 Storage Buckets Analysis </p>

In [4]:
# Initialize notebook
import sys
from pathlib import Path
sys.path.insert(0, Path("../../.."))

## Check Current Bucket Stats by pinging Object Store

In [3]:
from xaidar.s3Utils import decryptCredentials, initialize

# Initialize S3 client
import os
# credKey = ""
# credKey = os.getenv( "CRED_KEY" ) 
credPath =  Path( "../../../credentials.enc").resolve()
credDict = decryptCredentials( credKey, credPath )
client = initialize( "XChem", cred_dict=credDict)

# First Check Bucket Names
response = client.list_buckets()
bucketsList = [ bucket["Name"] for bucket in response['Buckets'] ]
print( "Available Buckets:", bucketsList )

Object Store Names:['XChem', 'MinIO']
Credentials Associated with each Object Store: ['endpoint_url', 'access_key', 'secret_key']
Available Buckets: ['pandda', 'sp21670-1', 'testbucket', 'tyt15771', 'xchem']


In [ ]:
# Get updated bucket Size
from xaidar.s3Utils import getBucketStatistic, bucketStorage, bucketObjCount
from xaidar.filesUtils import roundBytes
sizes = getBucketStatistic( client, bucketsList, bucketStorage, maxitems=1001)

print( "Bucket Sizes for first 100 000 Objects:")
for key, value in sizes.items():
    roundedSize, units = roundBytes(value)
    print( f"\t{key}: {roundedSize} {units}")
# getBucketStatistic( client, ["pandda", "testbucket"], bucketObjCount, maxitems=1001)


Bucket Sizes for first 100 000 Objects
	pandda: 3.48 TB
	sp21670-1: 0 Byte
	testbucket: 0 Byte
	tyt15771: 0 Byte
	xchem: 395.42 GB


## Bucket Storage Sizes

In [ ]:
# Get cached bucket Sizes
from xaidar.filesUtils import loadPickle



sizesDir = Path("../../../data/s3Sizes")
bucketNames = ["pandda", "xchem"]

bucketsContent = { bucket : { "storageSize":0, "filesCount": 0} for bucket  in bucketNames }

for bucketName in bucketNames:
    fragFilesDir = sizesDir / bucketName / "raw"
    for fragFilePath in fragFilesDir.iterdir():
        if fragFilePath.is_file():
            frag = loadPickle(fragFilePath)
            count = [ size for size in frag.values() if type(size) == int]
            filesCount = len( count )
            storageSize = sum( count )
            bucketsContent[bucketName]["storageSize"] += storageSize   
            bucketsContent[bucketName]["filesCount"]  += filesCount


print( "Done")

pd_storSize, pd_filsCount = bucketsContent["pandda"]["storageSize"], bucketsContent["pandda"]["filesCount"]  
xc_storSize, xc_filsCount = bucketsContent["xchem"]["storageSize"], bucketsContent["xchem"]["filesCount"] 

print( "For pandda:")
print( f"\tStorage Size: {pd_storSize},\tFiles Count: {pd_filsCount}")
print( "For xchem:")
print( f"\tStorage Size: {xc_storSize},\tFiles Count: {xc_filsCount}")

# Estimated Time: 2 m 17 s

In [18]:
from xaidar.filesUtils import roundBytes
pd_storSize, pd_filsCount = roundBytes(bucketsContent["pandda"]["storageSize"]), bucketsContent["pandda"]["filesCount"]  
xc_storSize, xc_filsCount = roundBytes(bucketsContent["xchem"]["storageSize"]), bucketsContent["xchem"]["filesCount"] 

print( "For pandda:")
print( f"\tStorage Size: {pd_storSize},\tFiles Count: {pd_filsCount}")
print( "For xchem:")
print( f"\tStorage Size: {xc_storSize},\tFiles Count: {xc_filsCount}")

For pandda:
	Storage Size: (8.47, 'TB'),	Files Count: 5115662
For xchem:
	Storage Size: (156.15, 'TB'),	Files Count: 129663436
